# Занятие 21. ML workflow. K ближайших соседей (kNN)

**ML workflow** (рабочий процесс ML) — это не «нажать кнопку "обучить"», а последовательность шагов: понять задачу → подготовить данные → обучить модель → честно проверить → понять ошибки.

Модель — лишь один фрагмент такого проекта. К хорошему результату приводит только точная постановка задачи, честная проверка и воспроизводимость эксперимента.

Чтобы разговор не был абстрактным, весь цикл работы над ML-задачей мы посмотрим на **методе k ближайших соседей (kNN)** — простой модели, на которой будут видны все этапы workflow.

К концу занятия вы будете:

- отличать обучение с учителем и без учителя, классификацию, регрессию и кластеризацию;
- описывать сквозной цикл ML-проекта и роль **train**/**validation**/**test** (обучающей, проверочной и тестовой частей данных);
- обучать kNN, сравнивать с **baseline** (простой моделью-ориентиром, см. п. 11) и один раз честно оценивать результат на тестовых данных.

## 1. Типы ML-задач

Процесс работы над ML-задачей начинают с определения **типа задачи** — от него зависят метрики, модели и способ проверки.

**Обучение без учителя** — в данных **нет готовых правильных ответов** (нет **меток**). Пример — **кластеризация**: разбить объекты на группы похожих без заранее заданных названий классов.

**Обучение с учителем** — для каждого объекта обучения известен **правильный ответ (метка)**. К этому типу относятся:

- **классификация** — предсказать категорию (спам/не спам, злокачественная/доброкачественная);
- **регрессия** — предсказать число (цена, температура, балл).

| Задача | Есть метки? | Что предсказываем? |
|---|---|---|
| кластеризация | нет | группы похожих объектов |
| классификация | да | категорию |
| регрессия | да | число |

Наш урок — **классификация с учителем**: по измерениям опухоли предсказываем её класс.

## 2. Сквозной цикл ML-проекта

Типичный **workflow** — не один запуск модели, а повторяемый цикл:

1. **Данные** — собрать таблицу, проверить объект наблюдения и момент прогноза (пп. 5–7).
2. **Инжиниринг признаков** — создать новые столбцы из уже имеющихся (п. 12).
3. **Предобработка** — заполнить пропуски, закодировать категории, привести числа к сравнимому масштабу; параметры таких преобразований считают **только на train** (пп. 13–14).
4. **Обучение модели** — обучить алгоритм на **train** (обучающей части данных).
5. **Метрики** — посчитать выбранную метрику на **validation** (проверочной части, где модель «не обучалась»).
6. **Сравнение гипотез** — при одном и том же протоколе выбрать лучший вариант: другой признак, модель или настройку (например, другое `k` в kNN).
7. **Анализ ошибок** — понять, где модель ошибается, и сформулировать следующую идею (пп. 17–18).

**Train**, **validation** и **test** — три части одной таблицы; зачем их три и как делить — в **п. 4**. После выбора решения один раз проверяют **test** (п. 19) и готовят мониторинг после внедрения (п. 20).


## 3. Метод k ближайших соседей (kNN)

Чтобы пройти весь цикл на живой модели, познакомимся с **методом k ближайших соседей (kNN)**. С его помощью будем решать задачу **классификации** на нашем датасете (хотя этот метод применим и к задаче регрессии).

**Признак** — один столбец таблицы (одно число про объект: радиус опухоли, возраст и т.д.). 

**Метка** — правильный класс объекта для обучения.

**Идея kNN:** для нового объекта находим **k самых похожих** объектов из данных для обучения и:
- **Классификация:**  «голосуем» — новый объект получает класс, который чаще встречается среди k соседей (в sklearm — `KNeighborsClassifier`).
- **Регрессия**: прогноз — среднее значение целевой переменной у k соседей (в sklearn — `KNeighborsRegressor`).

**Похожесть** измеряют **евклидовым расстоянием**: для двух объектов по каждому признаку берут разность, возводят в квадрат, складывают и извлекают корень. Чем **меньше** расстояние — тем объекты ближе в «пространстве признаков». Пример в коде ниже — по трём признакам.

kNN **не подбирает веса** (как, например, линейная модель) - стадии обучения у него по сути нет: когда в sklearn мы вызываем метод `.fit`, он просто **запоминает** все объекты обучающей выборки (`train`) и во время прогноза (`.predict`) каждый раз ищет соседей заново. Поэтому:

1. kNN чувствителен к **масштабу признаков**: если один столбец в тысячах, а другой в единицах, первый сильнее влияет на расстояние (проверим в п. 17–18).
2. Число соседей **`k`** — **гиперпараметр**: настройку задаём мы вручную, её не «выучивают» из данных. Как подбирать `k` — п. 16.

**Сквозной датасет.** Все примеры далее используют dataset `sklearn.datasets.load_breast_cancer`: по 30 числовым измерениям опухоли предсказываем класс **0** (злокачественная) или **1** (доброкачественная). Один объект — одна опухоль. Класс 0 записан первым в датасете — это не «плохая метка», просто так принято в этом наборе данных.


In [56]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X_all = data.data
y_all = data.target

print('Объектов:', X_all.shape[0], '| признаков:', X_all.shape[1])
print('Доля класса 1 (доброкачественные):', round(y_all.mean(), 3))
X_all.iloc[:, :4].head()

Объектов: 569 | признаков: 30
Доля класса 1 (доброкачественные): 0.627


,mean radius,mean texture,mean perimeter,mean area
0,17.99,10.38,122.80,1001.0
1,20.57,17.77,132.90,1326.0
2,19.69,21.25,130.00,1203.0
3,11.42,20.38,77.58,386.1
4,20.29,14.34,135.10,1297.0


In [57]:
# Евклидово расстояние между двумя опухолями по трём признакам
a = X_all.iloc[0, :3].to_numpy()
b = X_all.iloc[1, :3].to_numpy()
euclidean_dist = np.sqrt(np.sum((a - b) ** 2))
print('Признаки первой опухоли:', np.round(a, 2))
print('Признаки второй опухоли:', np.round(b, 2))
print('Евклидово расстояние (3 признака):', round(euclidean_dist, 2))

Признаки первой опухоли: [ 17.99  10.38 122.8 ]
Признаки второй опухоли: [ 20.57  17.77 132.9 ]
Евклидово расстояние (3 признака): 12.78


## 4. Train, validation и test

Прежде чем обучать модель, всю таблицу делят на три **непересекающиеся** части:

| Часть | Зачем нужна | Аналогия |
|---|---|---|
| **Train** | обучение модели и всех преобразований (для kNN — «запомнить» эти строки) | учебник, по которому готовимся |
| **Validation** | много раз сравнивать идеи и подбирать настройки (например, `k`) | пробные варианты домашки |
| **Test** | **один раз** проверить уже выбранное решение | контрольная работа |

**Почему не хватит train и test?** Если подбирать `k`, признаки и модель, глядя на test, test перестаёт быть честной проверкой — мы подстроимся под эти конкретные строки. Validation — «промежуточная площадка» для экспериментов.

Test похож на **запечатанный** вариант контрольной: после просмотра test нельзя возвращаться к обучению и «подкручивать» модель под те же объекты.

Для разбиения используем scikit-learn: **`train_test_split`** из `sklearn.model_selection`.

- **`random_state=42`** — фиксирует случайное разбиение: при тех же данных и том же числе получится тот же split (удобно сверять результаты с одноклассниками).
- **`stratify=y`** — сохраняет **доли классов** во всех частях. Если в полной таблице 63% доброкачественных, примерно столько же будет в train, validation и test. Иначе в одной части может случайно оказаться слишком мало злокачественных — и метрика станет нестабильной.

In [58]:
from sklearn.model_selection import train_test_split

# Стратифицируем по классу: доли классов сохранятся во всех трёх частях.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)
print('train / val / test:', len(X_train), len(X_val), len(X_test))


train / val / test: 341 114 114


## 5. От вопроса к ML-задаче

Вопрос «какие опухоли опасны?» ещё **не** ML-задача — он слишком общий. Нужно зафиксировать пять пунктов:

| Пункт | Вопрос | В нашем примере |
|---|---|---|
| **объект** | для кого один прогноз? | одна опухоль (одна строка таблицы) |
| **момент прогноза** | какая информация уже есть? | измерения биопсии; нет данных о лечении «через год» |
| **цель** | что предсказываем? | класс 0 или 1 |
| **действие** | что делаем с прогнозом? | при подозрении на класс 0 — дообследование |
| **метрика** | как измерим качество? | recall для класса 0 (п. 8) — доля найденных злокачественных |

**Сформулированная ML-задача:** «По 30 числовым признакам одной опухоли предсказать, злокачественная она (0) или доброкачественная (1); главное — не пропустить злокачественную».

## 6. Критерий успеха проекта

До моделирования договариваются, **что будет считаться успехом** — ещё до просмотра результатов.

**В нашей задаче:** успех — не пропустить злокачественную опухоль. Заранее можно записать, например:

- «recall для класса 0 на validation **не ниже 0,85**» (нашли не меньше 85% всех злокачественных; recall — п. 8);
- «доля **ложных тревог** не выше согласованного порога» (что это за ошибка — п. 8).

Если критерий придумать **после** эксперимента, легко «подогнать» его под удобную цифру — так делать нельзя.

## 7. Единица наблюдения и момент прогноза

**Объект наблюдения** — то, для чего делаем **один** прогноз. В `load_breast_cancer` это **одна опухоль** = **одна строка** таблицы.

**Момент прогноза** — граница между тем, что уже известно, и тем, чего ещё нет. В нашем датасете все 30 признаков — измерения **уже взятого** образца. Мы **не** используем:

- результат лечения через полгода;
- повторную биопсию через год.

Если добавить такой признак, модель получит **утечку данных** — информацию из «будущего», которой не будет в реальном прогнозе (подробнее п. 10).

## 8. Метрика следует из цены ошибок

Метрику выбирают **до** просмотра результатов модели. Сначала разберём, **какие бывают исходы** сравнения «истина vs прогноз».

### Четыре исхода: TP, FN, FP, TN

Для **каждого объекта** сравниваем истинный класс и прогноз модели. Возможны четыре исхода (стандартные сокращения в ML):

| | Модель сказала «злокач.» (0) | Модель сказала «доброкач.» (1) |
|---|---|---|
| **На самом деле злокач. (0)** | **TP** — верно нашли | **FN** — **пропустили** (опасно!) |
| **На самом деле доброкач. (1)** | **FP** — лишняя тревога | **TN** — верно успокоили |

Расшифровка:

- **TP** (true positive) — злокачественную верно назвали злокачественной.
- **FN** (false negative) — злокачественную ошибочно назвали доброкачественной. **Самая опасная ошибка** в нашей задаче.
- **FP** (false positive) — доброкачественную ошибочно назвали злокачественной (лишняя тревога).
- **TN** (true negative) — доброкачественную верно распознали.

Пропуск злокачественной (FN) опаснее лишней тревоги (FP), поэтому часто выбирают метрики, которые отражают именно FN.

### Метрики через TP, FN, FP

**Recall** (полнота) для класса 0 — какая **доля злокачественных** модель **нашла**:

recall = TP / (TP + FN)

*Пример:* было 10 злокачественных, модель нашла 9 → recall = 9/10 = 0,9.

**Precision** (точность) — какая доля срабатываний «злокачественная» **верна**:

precision = TP / (TP + FP)

*Пример:* модель 20 раз сказала «злокачественная», и только 15 раз была права → precision = 15/20 = 0,75.

**Accuracy** — доля **всех** верных ответов: (TP + TN) / (все объекты). Удобна для общего сравнения, но может скрыть провалы на редком классе.

**Про `pos_label`.** В `recall_score` слово «positive» **не** значит «хороший класс». **`pos_label`** — это «**про какой класс** считаем recall». Для злокачественных пишут `pos_label=0`, потому что класс 0 в этом датасете — злокачественная:

```python
recall_score(y_true, y_pred, pos_label=0)
```

## 9. Независимость объектов и способ разбиения

Как делить данные — зависит от того, **что будет новым** после внедрения модели:

| Сценарий | Как делить | Пример |
|---|---|---|
| объекты **независимы** | случайно | разные опухоли в нашем датасете |
| **один пациент — несколько строк** | все строки пациента в одной части | иначе модель «видела» этого пациента на train и проверяется на test |
| нужен прогноз **будущего** | по времени: прошлое → train, будущее → test | продажи по дням; признаки из будущего запрещены |



## 10. Утечка данных

**Утечка данных** — модель при обучении или настройке «увидела» то, чего **не будет** в момент реального прогноза. Метрика получается красивой, а в жизни модель работает хуже.

Типичные случаи:

- признак из **будущего** (результат лечения при первичном диагнозе);
- пропуски заполняли **средним по всей таблице** до split — test «подглядел» в train;
- подобрали `k` или признаки по **test**, а потом снова сообщили test-accuracy;
- **один пациент** и в train, и в test — модель уже «знакома» с похожими измерениями;
- при прогнозе будущего **перемешали** строки по времени.

**Правило:** всё, что считается по данным (среднее для пропусков, масштаб признаков), считаем **только на train**, а на validation и test только **применяем** уже посчитанное.

## 11. Baseline: с чем сравнивать kNN

**Baseline** — простая модель-**ориентир**: «что получится почти без умной логики». Если kNN не лучше baseline, усложнять модель рано.

Самый простой baseline для классификации — **всегда отвечать самым частым классом** в train. Например, если 63% опухолей доброкачественные, такая модель всегда говорит «доброкачественная» и получает accuracy ≈ 0,63 **без всякого kNN**.

**Accuracy** — доля верных ответов среди всех объектов.

В scikit-learn: `DummyClassifier(strategy='most_frequent')`. Сначала сравниваем kNN с этим baseline на validation — kNN должен быть **заметно лучше**.

In [59]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_acc = accuracy_score(y_val, dummy.predict(X_val))
print('Baseline (most_frequent) accuracy на validation:', round(dummy_acc, 3))
print('Доля большего класса в train:', round(y_train.mean() if y_train.mean() > 0.5 else 1 - y_train.mean(), 3))

Baseline (most_frequent) accuracy на validation: 0.623
Доля большего класса в train: 0.628


## 12. Инжиниринг признаков

**Инжиниринг признаков** — добавить **новый столбец**, вычисленный из уже имеющихся. Модель получает дополнительный сигнал без сбора новых данных.

Новые идеи **сравнивают на validation**. Test для выбора признаков **не** используют — иначе test перестаёт быть честной финальной проверкой (п. 4).

**Пример:** `radius_texture_ratio = mean radius / mean texture` — оба числа уже в строке, делим их. Статистику по всей таблице **не** считаем → утечки нет.

Если нужно **среднее для заполнения пропусков**, его считают **только на train** — это уже **предобработка** (пп. 13–14).

In [60]:
# Добавим осмысленный признак: отношение среднего радиуса к средней текстуре.
# Функция одинаково работает для train, validation и test — без утечки.
def add_features(df):
    out = df.copy()
    out['radius_texture_ratio'] = out['mean radius'] / out['mean texture']
    return out

X_train_fe = add_features(X_train)
X_val_fe = add_features(X_val)
X_test_fe = add_features(X_test)
print('Было признаков:', X_train.shape[1], '-> стало:', X_train_fe.shape[1])


Было признаков: 30 -> стало: 31


## 13. StandardScaler: fit на train, transform везде

**Предобработка** — привести данные к виду, удобному для модели. Для kNN важно **масштабирование**.

**`StandardScaler`** для каждого признака на train считает среднее и стандартное отклонение, затем преобразует значения: «отнять среднее, поделить на σ». После этого столбцы сравнимы по масштабу.

Два метода scikit-learn:

- **`fit`** — «выучить» параметры преобразования **только на train**;
- **`transform`** — применить **те же** параметры к train, validation и test.

На validation и test **`fit` вызывать нельзя** — иначе модель увидит статистику test при настройке (утечка).


## 14. Обучение kNN на масштабированных данных

**Порядок одного эксперимента:**

1. инжиниринг признаков;
2. `scaler.fit(X_train)` → `scaler.transform` для train, val, test;
3. `knn.fit(X_train_scaled, y_train)`;
4. метрика на validation;
5. **один раз** test (п. 19).


In [61]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# fit только на train, transform — на всех частях теми же параметрами
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_fe)
X_val_scaled = scaler.transform(X_val_fe)
X_test_scaled = scaler.transform(X_test_fe)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

val_pred = knn.predict(X_val_scaled)
val_accuracy = accuracy_score(y_val, val_pred)
print('kNN (k=5) accuracy на validation:', round(val_accuracy, 3))
print('Он должен превосходить baseline из п. 11:', round(dummy_acc, 3))

kNN (k=5) accuracy на validation: 0.947
Он должен превосходить baseline из п. 11: 0.623


## 15. Журнал экспериментов: подбор k

**Гиперпараметр `k`** — сколько соседей «голосуют». Слишком маленькое `k` — модель шумная; слишком большое — теряет локальные детали.

Перебираем несколько значений `k`, для каждого считаем метрику на **validation** и записываем в **журнал экспериментов** — таблицу с настройками и результатами. Лучшее `k` выбирают **только по validation**; test для подбора **не** смотрим (п. 4).

В журнале фиксируют: версию данных, `random_state`, модель, `k`, метрику, короткий комментарий.

**За один эксперимент меняют одну вещь** — иначе непонятно, что именно улучшило результат (сравнение «на равных» — п. 16).

In [62]:
# Реальный журнал: сравниваем baseline и kNN с разными k на одном и том же split.
runs = []
runs.append({'признаки': 'scaled', 'модель': 'Dummy(most_frequent)', 'k': None, 'val_acc': round(dummy_acc, 3)})
for k in [1, 3, 5, 11, 21]:
    model_k = KNeighborsClassifier(n_neighbors=k).fit(X_train_scaled, y_train)
    acc_k = accuracy_score(y_val, model_k.predict(X_val_scaled))
    runs.append({'признаки': 'scaled', 'модель': 'kNN', 'k': k, 'val_acc': round(acc_k, 3)})

experiments = pd.DataFrame(runs)
experiments


,признаки,модель,k,val_acc
0,scaled,Dummy(most_frequent),NaN,0.623
1,scaled,kNN,1.0,0.939
2,scaled,kNN,3.0,0.965
3,scaled,kNN,5.0,0.947
4,scaled,kNN,11.0,0.956
5,scaled,kNN,21.0,0.930


## 16. Одинаковый протокол сравнения

Две идеи сравнимы только **на равных**: один и тот же split, одна метрика, одни и те же строки.

**Протокол** — зафиксированный набор правил эксперимента. Сначала его записывают, потом меняют **одну** гипотезу.

Проверим: «масштабирование помогает kNN?» — один split, одно `k=5`, два варианта: с `StandardScaler` и без. Отличается **только** масштабирование.

In [63]:
# Одна гипотеза (масштабирование), один split, одна метрика.
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train_fe, y_train)
acc_raw = accuracy_score(y_val, knn_raw.predict(X_val_fe))

knn_scaled = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
acc_scaled = accuracy_score(y_val, knn_scaled.predict(X_val_scaled))

pd.DataFrame({
    'эксперимент': ['kNN без масштабирования', 'kNN со StandardScaler'],
    'val_accuracy': [round(acc_raw, 3), round(acc_scaled, 3)],
})


,эксперимент,val_accuracy
0,kNN без масштабирования,0.930
1,kNN со StandardScaler,0.947


## 17. Confusion matrix на validation

Средняя метрика (accuracy, recall) **скрывает детали**. На validation строят **матрицу ошибок** (`confusion_matrix`) — та же таблица TP/FN/FP/TN из п. 8, но с **числами** вместо букв:

- **строки** — истинный класс (как было на самом деле);
- **столбцы** — что **предсказала** модель.

Код ниже выводит матрицу с подписями. Для recall по классу 0 смотрим **первую строку** (ячейки TP и FN).


## 18. Разбор ошибок по группам

Полезно разбить validation на **группы** (крупные / мелкие опухоли) и посмотреть **долю ошибок в каждой группе отдельно**. Эти доли **не обязаны** в сумме давать 1 — это доли **внутри** своей группы.


In [64]:
# Матрица ошибок и recall для класса 0 (злокачественная)
cm = confusion_matrix(y_val, val_pred)
print('confusion_matrix (строки — истина, столбцы — прогноз):')
print(cm)

cm_labeled = pd.DataFrame(
    cm,
    index=['истина 0 (злокач.)', 'истина 1 (доброкач.)'],
    columns=['прогноз 0', 'прогноз 1'],
)
display(cm_labeled)

tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
print('TP, FN, FP, TN (положительный класс 0):', tp, fn, fp, tn)
print('Recall злокачественных (pos_label=0):', round(recall_score(y_val, val_pred, pos_label=0), 3))

median_radius = X_train['mean radius'].median()
group = np.where(X_val['mean radius'] >= median_radius, 'крупные', 'мелкие')
error_table = pd.DataFrame({
    'группа': group,
    'ошибка': (val_pred != y_val.to_numpy()).astype(int),
})
error_rate = error_table.groupby('группа', observed=True)['ошибка'].mean().round(3)
print('Доля ошибок внутри каждой группы на validation (не суммируется в 1):')
print(error_rate.to_string())

confusion_matrix (строки — истина, столбцы — прогноз):
[[37  6]
 [ 0 71]]


,прогноз 0,прогноз 1
истина 0 (злокач.),37,6
истина 1 (доброкач.),0,71


TP, FN, FP, TN (положительный класс 0): 37 6 0 71
Recall злокачественных (pos_label=0): 0.86
Доля ошибок внутри каждой группы на validation (не суммируется в 1):
группа
крупные    0.081
мелкие     0.019


## 19. Финальный test и внедрение

Когда на validation выбрали лучшее `k`, признаки и преобразования — **фиксируют** весь процесс и **один раз** считают метрики на **test**. После этого test **больше не трогают** для улучшения этой версии модели.

После внедрения модель всё равно нужно **наблюдать (мониторинг)**:

- не изменились ли распределения признаков;
- нет ли пропусков или новых категорий;
- не упало ли качество на новых данных;
- хватает ли скорости системы.

Данные и мир меняются — модель, которая хорошо работала вчера, завтра может устареть.

In [65]:
# Выбрали kNN с лучшим k по validation. Один раз проверяем на test и больше его не трогаем.
knn_runs = experiments[experiments['модель'] == 'kNN']
best_k = int(knn_runs.loc[knn_runs['val_acc'].idxmax(), 'k'])
final_model = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_scaled, y_train)
test_pred = final_model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, test_pred)
print(f'Финальная accuracy kNN (k={best_k}) на test:', round(test_acc, 3))
print('Recall злокачественных на test:', round(recall_score(y_test, test_pred, pos_label=0), 3))
print('Baseline accuracy на validation был:', round(dummy_acc, 3))

Финальная accuracy kNN (k=3) на test: 0.947
Recall злокачественных на test: 0.929
Baseline accuracy на validation был: 0.623


## 20. Чек-лист эксперимента

Краткая памятка по циклу из п. 2:

1. Тип задачи и постановка (пп. 1, 5–8).
2. Разбиение train / validation / test без утечек (пп. 4, 9–10).
3. Baseline (п. 11).
4. Признаки, предобработка, модель — параметры преобразований только на train (пп. 12–14).
5. Сравнение идей и подбор `k` на validation, журнал экспериментов (пп. 15–16).
6. Анализ ошибок (пп. 17–18).
7. Один раз test и мониторинг после внедрения (п. 19).
